# Deployment threshold sensitivity

Compare per-sensor source presence at strict score cutoffs 0.3, 0.5 and 0.7.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display
from analysis_paths import connect_analysis, output_path, bundle_fingerprint
OUT = output_path('out/threshold_sensitivity')
FIG = output_path('figures/threshold_sensitivity')
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
THRESHOLDS = (0.3, 0.5, 0.7)
CLASSES = ['vehicle','honking','aircraft','siren','human','bark','bird','church','music','wind','rain']
checks = {}
def check(name, condition):
    checks[name] = bool(condition)
    assert condition, name
db = connect_analysis()


## Location sensitivity

Map ratios exclude NULL scores; overall-chart ratios retain all rows in their denominators. Sensor means weight each sensor equally.


In [ ]:
parts = ['COUNT(*) AS total_rows', 'MIN(time) AS first_time', 'MAX(time) AS last_time',
         'COUNT(*) FILTER (WHERE time IS NULL) AS null_time_rows']
for cls in CLASSES:
    parts += [f'COUNT("{cls}") AS "{cls}_valid"',
              f'COUNT(*) FILTER (WHERE "{cls}" = 0.5) AS "{cls}_ties"',
              f'COUNT(*) FILTER (WHERE "{cls}" < 0 OR "{cls}" > 1 OR isnan("{cls}")) AS "{cls}_invalid"']
    for k, threshold in enumerate(THRESHOLDS):
        parts += [f'SUM(("{cls}" > {threshold})::BIGINT) AS "{cls}_hits_{k}"',
                  f'100.0*AVG(("{cls}" > {threshold})::DOUBLE) AS "{cls}_pct_{k}"']
query = 'SELECT sensor_id, ' + ', '.join(parts) + ' FROM sources GROUP BY sensor_id ORDER BY sensor_id'
wide = db.execute(query).df()
rows = []
for sensor in wide.to_dict('records'):
    for cls in CLASSES:
        for k, threshold in enumerate(THRESHOLDS):
            hits = sensor[f'{cls}_hits_{k}']
            rows.append(dict(sensor_id=sensor['sensor_id'], **{'class':cls}, threshold=threshold,
                total_rows=sensor['total_rows'], valid_score_rows=sensor[f'{cls}_valid'],
                detections=hits, map_presence_pct=sensor[f'{cls}_pct_{k}'],
                chart_presence_pct=100*hits/sensor['total_rows']))
location = pd.DataFrame(rows)
wide.to_csv(OUT/'deployment_input_audit.csv', index=False)
base = location[location.threshold.eq(0.5)][['sensor_id','class','map_presence_pct']].rename(columns={'map_presence_pct':'baseline_pct'})
location = location.merge(base, on=['sensor_id','class'], validate='many_to_one')
location['change_pp'] = location.map_presence_pct-location.baseline_pct
location.to_csv(OUT/'location_sensitivity.csv', index=False)
hit_matrix = location.pivot(index=['sensor_id','class'], columns='threshold', values='detections')
check('detection counts monotone with threshold', bool((hit_matrix.diff(axis=1).iloc[:,1:] <= 0).all().all()))
check('same denominators at every threshold', bool(location.groupby(['sensor_id','class'])[['total_rows','valid_score_rows']].nunique().eq(1).all().all()))

# Independent reproduction of the existing map and overall-chart SQL at 0.5.
baseline_parts = []
for cls in CLASSES:
    baseline_parts += [f'100.0*AVG(({cls} > 0.5)::DOUBLE) AS "{cls}_map"',
                       f'100.0*SUM(({cls} > 0.5)::INTEGER)/COUNT(*) AS "{cls}_chart"']
baseline = db.execute('SELECT sensor_id, '+', '.join(baseline_parts)+' FROM sources GROUP BY sensor_id ORDER BY sensor_id').df().set_index('sensor_id')
for cls in CLASSES:
    actual = location[(location['class']==cls)&location.threshold.eq(0.5)].set_index('sensor_id').loc[baseline.index]
    check('map baseline '+cls, np.allclose(actual.map_presence_pct, baseline[f'{cls}_map'], equal_nan=True))
    check('chart baseline '+cls, np.allclose(actual.chart_presence_pct, baseline[f'{cls}_chart'], equal_nan=True))
summary_rows = []
for (cls, threshold), g in location.groupby(['class','threshold'], sort=False):
    rho = spearmanr(g.baseline_pct, g.map_presence_pct).statistic if g.baseline_pct.nunique()>1 and g.map_presence_pct.nunique()>1 else np.nan
    summary_rows.append(dict(**{'class':cls}, threshold=threshold, sensors=len(g),
        mean_presence_pct=g.chart_presence_pct.mean(), mean_change_pp=g.change_pp.mean(),
        max_abs_change_pp=g.change_pp.abs().max(), rank_spearman=rho))
location_summary = pd.DataFrame(summary_rows)
location_summary.to_csv(OUT/'location_summary.csv', index=False)
display(location_summary.round(4))
print('Deployment:', len(wide), 'sensors;', int(wide.total_rows.sum()), 'rows;', wide.first_time.min(), 'to', wide.last_time.max())
print('NULL scores:', {c:int((wide.total_rows-wide[f'{c}_valid']).sum()) for c in CLASSES})
print('Exact 0.5 ties:', {c:int(wide[f'{c}_ties'].sum()) for c in CLASSES})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
colors = {0.3:'#0072B2', 0.5:'#333333', 0.7:'#D55E00'}
x = np.arange(len(CLASSES))
for k, threshold in enumerate(THRESHOLDS):
    s = location_summary[location_summary.threshold.eq(threshold)].set_index('class').loc[CLASSES]
    axes[0].bar(x+(k-1)*0.25, s.mean_presence_pct, width=0.25, color=colors[threshold], label=str(threshold))
for threshold, offset in [(0.3,-0.12),(0.7,0.12)]:
    for i, cls in enumerate(CLASSES):
        g = location[(location['class']==cls)&location.threshold.eq(threshold)]
        axes[1].scatter(np.full(len(g),i+offset), g.change_pp, s=12, alpha=0.55, color=colors[threshold], label=str(threshold) if i==0 else None)
axes[0].set_ylabel('Mean sensor presence (%)')
axes[0].set_title('Absolute ratios; original sensor weighting')
axes[1].set_ylabel('Sensor presence change from 0.5 (pp)')
axes[1].set_title('Each point represents one sensor')
axes[1].axhline(0,color='gray',linewidth=0.8)
for ax in axes:
    ax.set_xticks(x, CLASSES, rotation=45, ha='right')
    ax.legend(title='Threshold')
    ax.grid(axis='y',alpha=0.2)
fig.savefig(FIG/'location_threshold_sensitivity.png', dpi=180)
plt.show()

In [ ]:
check('38 sensors and 56923211 fragments', len(wide)==38 and wide.total_rows.sum()==56923211)
check('no missing timestamps', wide.null_time_rows.sum()==0)
for cls in CLASSES:
    check(cls+' valid scores', wide[cls+'_invalid'].sum()==0)
    check(cls+' no missing scores', wide[cls+'_valid'].sum()==wide.total_rows.sum())
(OUT/'checks.json').write_text(json.dumps({'checks': checks, 'thresholds': THRESHOLDS,
    'data_manifest_sha256': bundle_fingerprint()}, indent=2), encoding='utf-8')
db.close()
